<a href="https://colab.research.google.com/github/ashhwiithac22/Adversarial_Patch_Detection_for_Military_Drones/blob/main/AI_Firewall_For_Drones_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset Import

In [1]:
# Cell 1: Simplified mount
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
# Check if already mounted
import os
if os.path.exists('/content/drive/MyDrive'):
    print("✅ Drive already mounted")
else:
    from google.colab import drive
    drive.mount('/content/drive')

✅ Drive already mounted


In [3]:
# Cell 1: Mount Google Drive with retry
from google.colab import drive
import time

max_attempts = 3
for attempt in range(max_attempts):
    try:
        drive.mount('/content/drive', force_remount=True)
        print("✅ Drive mounted successfully")
        break
    except Exception as e:
        print(f"Attempt {attempt + 1} failed: {e}")
        if attempt < max_attempts - 1:
            print("Retrying in 5 seconds...")
            time.sleep(5)
        else:
            print("❌ All mount attempts failed")
            raise

Mounted at /content/drive
✅ Drive mounted successfully


In [4]:
# Cell 2: Create project folder
import os

# Create project folder
!mkdir -p /content/drive/MyDrive/AI_Firewall_For_Drones

# Change to project directory
%cd /content/drive/MyDrive/AI_Firewall_For_Drones

# Verify
!pwd

/content/drive/MyDrive/AI_Firewall_For_Drones
/content/drive/MyDrive/AI_Firewall_For_Drones


In [14]:
# Cell 3: Install all dependencies
!pip install torch torchvision torchattacks --quiet
!pip install kaggle pandas numpy scikit-learn tqdm --quiet
!pip install opencv-python pillow imagehash --quiet
!pip install matplotlib seaborn --quiet

# Verify
import torch
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

✅ PyTorch: 2.10.0+cu128
✅ CUDA available: True
✅ GPU: Tesla T4


In [15]:
# Cell 4: Set Kaggle API Token directly (No file upload needed!)
import os

# Set your API token as environment variable
os.environ['KAGGLE_USERNAME'] = 'Ashwitha C'  # Replace with your Kaggle username
os.environ['KAGGLE_KEY'] = 'KGAT_ba34d38359b59577be4dc6131fefb151'

# Verify
print("✅ Kaggle API configured")
!kaggle --version

✅ Kaggle API configured
Kaggle CLI 2.0.0


###Clean images without adversarial patches

In [35]:
# Cell: Create clean folder and copy images
import os
import random
import shutil
from tqdm import tqdm

DRIVE_PATH = "/content/drive/MyDrive/AI_Firewall_For_Drones"
DATASET_PATH = f"{DRIVE_PATH}/dataset_full/dataset"
CLEAN_PATH = f"{DATASET_PATH}/clean"

print("="*60)
print("📁 CREATING CLEAN FOLDER")
print("="*60)

# Create clean folder
os.makedirs(CLEAN_PATH, exist_ok=True)
print(f"✅ Created: {CLEAN_PATH}")

# Collect all images from train, test, validation
all_images = []

for folder in ["train", "test", "validation"]:
    folder_path = os.path.join(DATASET_PATH, folder)
    if os.path.exists(folder_path):
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                    all_images.append(os.path.join(root, file))
        print(f"   ✅ Found images in {folder}/")

print(f"\n✅ Total images found: {len(all_images)}")

# Select 2500 images
random.seed(42)
selected = random.sample(all_images, min(2500, len(all_images)))

print(f"\n📸 Copying {len(selected)} images to clean folder...")

for idx, img_path in enumerate(tqdm(selected)):
    ext = os.path.splitext(img_path)[1]
    filename = f"mil_{idx+1:04d}{ext}"
    dest_path = os.path.join(CLEAN_PATH, filename)
    shutil.copy2(img_path, dest_path)

print(f"\n✅ {len(os.listdir(CLEAN_PATH))} images saved!")
print(f"📁 Location: {CLEAN_PATH}")

# Show what's now in dataset folder
print("\n📁 Updated contents of dataset_full/dataset:")
for item in os.listdir(DATASET_PATH):
    item_path = os.path.join(DATASET_PATH, item)
    if os.path.isdir(item_path):
        count = len(os.listdir(item_path))
        print(f"   📁 {item}/ : {count} files")

📁 CREATING CLEAN FOLDER
✅ Created: /content/drive/MyDrive/AI_Firewall_For_Drones/dataset_full/dataset/clean
   ✅ Found images in train/
   ✅ Found images in test/
   ✅ Found images in validation/

✅ Total images found: 16189

📸 Copying 2500 images to clean folder...


100%|██████████| 2500/2500 [01:10<00:00, 35.25it/s]



✅ 2500 images saved!
📁 Location: /content/drive/MyDrive/AI_Firewall_For_Drones/dataset_full/dataset/clean

📁 Updated contents of dataset_full/dataset:
   📁 test/ : 10 files
   📁 train/ : 10 files
   📁 validation/ : 6 files
   📁 clean/ : 2500 files


### Created Adversarial images with patches using OpenCV(image manipulation)

In [37]:
# ============================================================
# CELL 1: GENERATE ADVERSARIAL IMAGES (PHYSICAL PATCHES)
# ============================================================
# This code reads clean images and applies physical adversarial patches
# like tape, camouflage, checkerboard patterns, and noise patches.
# ============================================================

import os
import cv2
import random
import numpy as np
from tqdm import tqdm
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Set paths
DRIVE_PATH = "/content/drive/MyDrive/AI_Firewall_For_Drones"
DATASET_PATH = f"{DRIVE_PATH}/dataset_full/dataset"
CLEAN_PATH = f"{DATASET_PATH}/clean"
ADV_PATH = f"{DATASET_PATH}/adversarial"

print("="*70)
print("⚔️ ADVERSARIAL IMAGE GENERATION")
print("="*70)
print(f"Source (Clean Images): {CLEAN_PATH}")
print(f"Destination (Adversarial): {ADV_PATH}")
print("="*70)

# Create adversarial folder
os.makedirs(ADV_PATH, exist_ok=True)

# Get all clean images
clean_files = [f for f in os.listdir(CLEAN_PATH) if f.endswith(('.jpg', '.jpeg', '.png'))]
print(f"\n✅ Found {len(clean_files)} clean images")

# ============================================================
# ATTACK FUNCTIONS (Physical Adversarial Patches)
# ============================================================

def add_white_tape(img):
    """Add white tape stripes (like military markings)"""
    h, w = img.shape[:2]
    for i in range(random.randint(2, 4)):
        y = random.randint(h//4, 3*h//4)
        cv2.rectangle(img, (50, y), (w-50, y+random.randint(10, 15)), (255, 255, 255), -1)
    return img

def add_yellow_tape(img):
    """Add yellow caution tape patterns"""
    h, w = img.shape[:2]
    for x in range(50, w-50, random.randint(30, 50)):
        cv2.rectangle(img, (x, h//2-15), (x+25, h//2+15), (0, 255, 255), -1)
    return img

def add_red_patch(img):
    """Add red square patch (adversarial marker)"""
    h, w = img.shape[:2]
    size = random.randint(35, 65)
    x = random.randint(20, w-size-20)
    y = random.randint(20, h-size-20)
    cv2.rectangle(img, (x, y), (x+size, y+size), (0, 0, 255), -1)
    return img

def add_woodland_camouflage(img):
    """Add woodland camouflage pattern (green, brown, black spots)"""
    h, w = img.shape[:2]
    colors = [(34, 139, 34), (101, 67, 33), (50, 50, 50)]  # Green, Brown, Black
    for _ in range(250):
        x = random.randint(0, w)
        y = random.randint(0, h)
        r = random.randint(15, 35)
        color = random.choice(colors)
        cv2.circle(img, (x, y), r, color, -1)
    return img

def add_desert_camouflage(img):
    """Add desert camouflage pattern (tan, brown, beige spots)"""
    h, w = img.shape[:2]
    colors = [(194, 178, 128), (210, 180, 140), (160, 120, 80)]  # Tan, Beige, Brown
    for _ in range(250):
        x = random.randint(0, w)
        y = random.randint(0, h)
        r = random.randint(15, 35)
        color = random.choice(colors)
        cv2.circle(img, (x, y), r, color, -1)
    return img

def add_checkerboard_patch(img):
    """Add classic adversarial checkerboard pattern"""
    h, w = img.shape[:2]
    size = 40
    # Position in top-left or bottom-right area
    start_x = random.choice([30, w-130])
    start_y = random.choice([30, h-130])
    for i in range(0, 100, size):
        for j in range(0, 100, size):
            if (i//size + j//size) % 2 == 0:
                cv2.rectangle(img, (start_x+i, start_y+j), (start_x+i+size, start_y+j+size), (0, 0, 0), -1)
            else:
                cv2.rectangle(img, (start_x+i, start_y+j), (start_x+i+size, start_y+j+size), (255, 255, 255), -1)
    return img

def add_noise_patch(img):
    """Add random noise patch (digital perturbation)"""
    h, w = img.shape[:2]
    patch_size = random.randint(50, 80)
    x = random.randint(20, w-patch_size-20)
    y = random.randint(20, h-patch_size-20)
    noise = np.random.randint(0, 255, (patch_size, patch_size, 3), dtype=np.uint8)
    img[y:y+patch_size, x:x+patch_size] = noise
    return img

def add_red_white_stripes(img):
    """Add red and white alternating stripes (like research paper)"""
    h, w = img.shape[:2]
    for i in range(3):
        y = h//3 + i*50
        cv2.rectangle(img, (40, y), (w-40, y+15), (0, 0, 255), -1)
        cv2.rectangle(img, (40, y+20), (w-40, y+35), (255, 255, 255), -1)
    return img

# ============================================================
# ATTACK DISTRIBUTION (Matches Research Paper)
# ============================================================
attack_types = [
    ("white_tape", add_white_tape, 350),
    ("yellow_tape", add_yellow_tape, 350),
    ("red_patch", add_red_patch, 350),
    ("woodland_camo", add_woodland_camouflage, 350),
    ("desert_camo", add_desert_camouflage, 350),
    ("checkerboard", add_checkerboard_patch, 250),
    ("noise_patch", add_noise_patch, 250),
    ("red_white_stripes", add_red_white_stripes, 250)
]

print("\n📊 Attack Distribution:")
for name, func, count in attack_types:
    print(f"   - {name}: {count} images")

# ============================================================
# GENERATE ADVERSARIAL IMAGES
# ============================================================
print("\n📸 Generating adversarial images...")

successful = 0
failed = 0

# Create a queue of attack assignments
attack_queue = []
for name, func, count in attack_types:
    attack_queue.extend([(name, func)] * count)

# Shuffle to randomize distribution
random.shuffle(attack_queue)

# Ensure we have enough attacks for all images
while len(attack_queue) < len(clean_files):
    attack_queue.extend(attack_queue[:len(clean_files) - len(attack_queue)])

# Process each clean image
for idx, clean_file in enumerate(tqdm(clean_files, desc="Processing")):
    try:
        # Read clean image
        clean_path = os.path.join(CLEAN_PATH, clean_file)
        img = cv2.imread(clean_path)

        if img is None:
            print(f"   Warning: Could not read {clean_file}")
            failed += 1
            continue

        # Get attack for this image
        attack_name, attack_func = attack_queue[idx % len(attack_queue)]

        # Apply adversarial patch
        adv_img = attack_func(img.copy())

        # Save adversarial image
        name, ext = os.path.splitext(clean_file)
        adv_filename = f"{name}_{attack_name}{ext}"
        adv_path = os.path.join(ADV_PATH, adv_filename)
        cv2.imwrite(adv_path, adv_img)

        successful += 1

    except Exception as e:
        print(f"   Error on {clean_file}: {e}")
        failed += 1

# ============================================================
# VERIFICATION
# ============================================================
print("\n" + "="*70)
print("✅ GENERATION COMPLETE!")
print("="*70)

adv_count = len(os.listdir(ADV_PATH))
print(f"\n📊 Statistics:")
print(f"   ✅ Clean images processed: {successful}")
print(f"   ❌ Failed: {failed}")
print(f"   📁 Adversarial images created: {adv_count}")

print(f"\n📂 Location in Google Drive:")
print(f"   {ADV_PATH}")

# Show attack distribution in results
print("\n📊 Generated Attack Types:")
attack_counts = {}
for f in os.listdir(ADV_PATH):
    for name, _, _ in attack_types:
        if name in f:
            attack_counts[name] = attack_counts.get(name, 0) + 1
            break

for name, count in sorted(attack_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"   - {name}: {count} images")

# Show sample files
print("\n📸 Sample adversarial images:")
for f in os.listdir(ADV_PATH)[:5]:
    print(f"   - {f}")

print("\n" + "="*70)
print("🚀 READY FOR MODEL TRAINING!")
print("="*70)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⚔️ ADVERSARIAL IMAGE GENERATION
Source (Clean Images): /content/drive/MyDrive/AI_Firewall_For_Drones/dataset_full/dataset/clean
Destination (Adversarial): /content/drive/MyDrive/AI_Firewall_For_Drones/dataset_full/dataset/adversarial

✅ Found 2500 clean images

📊 Attack Distribution:
   - white_tape: 350 images
   - yellow_tape: 350 images
   - red_patch: 350 images
   - woodland_camo: 350 images
   - desert_camo: 350 images
   - checkerboard: 250 images
   - noise_patch: 250 images
   - red_white_stripes: 250 images

📸 Generating adversarial images...


Processing: 100%|██████████| 2500/2500 [01:10<00:00, 35.59it/s]



✅ GENERATION COMPLETE!

📊 Statistics:
   ✅ Clean images processed: 2500
   ❌ Failed: 0
   📁 Adversarial images created: 2500

📂 Location in Google Drive:
   /content/drive/MyDrive/AI_Firewall_For_Drones/dataset_full/dataset/adversarial

📊 Generated Attack Types:
   - desert_camo: 350 images
   - yellow_tape: 350 images
   - white_tape: 350 images
   - red_patch: 350 images
   - woodland_camo: 350 images
   - red_white_stripes: 250 images
   - checkerboard: 250 images
   - noise_patch: 250 images

📸 Sample adversarial images:
   - mil_0001_desert_camo.jpeg
   - mil_0002_desert_camo.jpeg
   - mil_0003_yellow_tape.jpeg
   - mil_0004_white_tape.jpeg
   - mil_0005_red_white_stripes.jpeg

🚀 READY FOR MODEL TRAINING!


# Model Training